# Testing script of the SimpleCNN model

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

# Local imports
from cnn import SimpleCNN, SimpleAdaptiveCNN, TestCNN
from data.load_cifar import CIFAR10
from train import train_model
from analysis.evaluation import evaluate_model
from data.load_gtsrb import GTSRB_CLASSES, GTSRB


# https://medium.com/@BurtMcGurt/a-practical-guide-to-data-augmentation-in-pytorch-with-examples-and-visualizations-761ad5c2a903
# https://docs.pytorch.org/vision/0.21/transforms.html
import torchvision.transforms as transforms
from torchvision.transforms import v2

import matplotlib.pyplot as plt
import numpy as np

## Experimental Area
Run the cnns on different padding (0, 1, 3) and kernel size as well (3x3, 5x5, 7x7) with Epoch 10 (so six runs)

In [ ]:
# Device selection (prep for potential GPU usage in e.g Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Data Augmentation
train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomCrop(64, padding=4), # padding is ofc applied dbefore cropping randomly
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3)
])
#train_transforms = None 
val_transforms = None

training_path = "../data/GTSRB/Final_Training/Images"
testing_path = "../data/GTSRB/Final_Test/Images"

# Create datasets
train_dataset = GTSRB(base_path=training_path, split="train", transform=train_transforms)
test_dataset  = GTSRB(base_path=testing_path, split="test", transform=None)

# Class names in GTSRB
class_names = list(GTSRB_CLASSES.values())

# Create dataloaders
trainloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valloader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

# check the "batching"
images, labels = next(iter(trainloader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

#Using device: cpu
#Loading GTSRB (train) from ../data/GTSRB/Final_Training/Images...
#Loading GTSRB (test) from ../data/GTSRB/Final_Test/Images...
#Train samples: 39209
#Test samples: 12630
#Image batch shape: torch.Size([128, 3, 64, 64])
#Label batch shape: torch.Size([128])
#Image dtype: torch.float32
#Label dtype: torch.int64

In [ ]:
''' 
The commented out ones obviously dont work with 32x32 with out pooling techniques as it will be 0x0 at the end of the conv network
                    "Test_CNN_pad0_kernelsize5": 
                   
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 5,
                       padding = 0),
                    

                    "Test_CNN_pad0_kernelsize7": 
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 7,
                       padding = 0),

                    "Test_CNN_pad1_kernelsize7": 
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 7,
                       padding = 1),
'''

In [ ]:
in_channels = 3
num_classes = len(GTSRB_CLASSES)
input_size = 64
base_channels = 32
channel_multiplier = 2
epochs=10


model_selection = {
                    "Test_CNN_pad0_kernelsize3": 
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,
                       input_size=64,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 3,
                       padding = 0),

                    "Test_CNN_pad1_kernelsize3": 
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,
                       input_size=64,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 3,
                       padding = 1),

                    "Test_CNN_pad1_kernelsize5": 
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,                       
                       input_size=64,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 5,
                       padding = 1),

                    "Test_CNN_pad3_kernelsize3": 
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,                       
                       input_size=64,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 3,
                       padding = 3),

                    "Test_CNN_pad3_kernelsize5": 
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,
                       input_size=64,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 5,
                       padding = 3),

                    "Test_CNN_pad3_kernelsize7": 
                   TestCNN(
                       in_channels=in_channels,
                       num_classes=num_classes,
                       base_channels=base_channels,
                       input_size=64,
                       channel_multiplier=channel_multiplier,
                       dropout_rate=0.1,
                       kernel_size = 7,
                       padding = 3)
                    }

for model_name, model in model_selection.items():
    print(f"Training model: {model_name}")
    # Set the model to the device (GPU/CPU)
    model = model.to(device)
    print(model)

    # Prepare Loss function and optimizer
    criterion = nn.CrossEntropyLoss() # loss function cross entropy that will be used for calculating the loss during training so that the optimizer can update the weights accordingly (remember backpropagation!)

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    print("Starting training...")
    # Train the model
    trained_model, train_accuracies, val_accuracies = train_model(
        model=model,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epochs=epochs
        )
    print("Training completed.")    

    # Save the trained model and check for exisiting files/dirs 
    Path("cnn_trained_models").mkdir(exist_ok=True)
    file_name = f"cnn_trained_models/{model_name.lower()}_cifar10.pth"
    
    # Check if file exists and append version number
    base_path = Path(file_name)
    if base_path.exists():
        version = 1
        while (base_path.parent / f"{base_path.stem}_v{version}{base_path.suffix}").exists():
            version += 1
        file_name = str(base_path.parent / f"{base_path.stem}_v{version}{base_path.suffix}")

    torch.save(trained_model.state_dict(), file_name)
    print("Saved trained model to:", file_name)
    
    print("Evaluating trained model on validation set...")
    trained_model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = trained_model(images)
            _, preds = outputs.max(1)

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    print(f"Final validation accuracy: {correct / total:.4f}")

    evaluate_model(trained_model, valloader, device=device, class_names=class_names)

    del trained_model

## CIFAR related Area

### Training Area

In [ ]:
# Device selection (prep for potential GPU usage in e.g Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Data Augmentation
train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomCrop(32, padding=4), # padding is ofc applied dbefore cropping randomly
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3)
])
#train_transforms = None
val_transforms = None

# Create datasets
train_dataset = CIFAR10(split="train", transform=train_transforms)
test_dataset  = CIFAR10(split="test", transform=val_transforms)

# Class names in CIFAR-10
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck" ]

# Create dataloaders
trainloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valloader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

# check the "batching"
images, labels = next(iter(trainloader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

In [ ]:
in_channels = 3
num_classes = 10
input_size = 32
base_channels = 32
channel_multiplier = 2
epochs=40

model_selection = {"SimpleCNN_40epochs_01_dropit": SimpleCNN(
                                in_channels=in_channels,
                                num_classes=num_classes,
                                base_channels=base_channels,
                                channel_multiplier=channel_multiplier,
                                dropout_rate=0.1)}

for model_name, model in model_selection.items():
    print(f"Training model: {model_name}")
    # Set the model to the device (GPU/CPU)
    model = model.to(device)
    print(model)

    # Prepare Loss function and optimizer
    criterion = nn.CrossEntropyLoss() # loss function cross entropy that will be used for calculating the loss during training so that the optimizer can update the weights accordingly (remember backpropagation!)

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    print("Starting training...")
    # Train the model
    trained_model, train_accuracies, val_accuracies = train_model(
        model=model,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epochs=epochs
        )
    print("Training completed.")    

    # Save the trained model and check for exisiting files/dirs 
    Path("cnn_trained_models").mkdir(exist_ok=True)
    file_name = f"cnn_trained_models/{model_name.lower()}_cifar10.pth"
    
    # Check if file exists and append version number
    base_path = Path(file_name)
    if base_path.exists():
        version = 1
        while (base_path.parent / f"{base_path.stem}_v{version}{base_path.suffix}").exists():
            version += 1
        file_name = str(base_path.parent / f"{base_path.stem}_v{version}{base_path.suffix}")

    torch.save(trained_model.state_dict(), file_name)
    print("Saved trained model to:", file_name)
    
    print("Evaluating trained model on validation set...")
    trained_model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = trained_model(images)
            _, preds = outputs.max(1)

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    print(f"Final validation accuracy: {correct / total:.4f}")

    evaluate_model(trained_model, valloader, device=device, class_names=class_names)

    del trained_model

### Show here the training and validation times of our models 
Dropout at 0.5, only things changed is augmentation on/off and the comparison between the two CNN models over number of Epochs

In [ ]:
train_val_times = {   
    #/cnn_trained_models/Ten_epochs and /cnn_training_curves/Ten_epochs
    "SimpleCNN_10Epochs_noAug": {
        "Total Training Time": 338.53 ,
        "Total Validation Time": 43.4 
        } ,
    "SimpleAdaptiveCNN_10Epochs_noAug":{
        "Total Training Time"  : 338.12 ,
        "Total Validation Time": 42.64
        },
    "SimpleCNN_10Epochs_Aug": {
        "Total Training Time"  : 417.16,
        "Total Validation Time": 50.60
        },
    "SimpleAdaptiveCNN_10Epochs_Aug" : {
        "Total Training Time" : 454.24,
        "Total Validation Time": 51.46,
        },

    # /cnn_trained_models/Fifteen_epochs and /cnn_training_curves/Fifteen_epochs
    "SimpleCNN_15Epochs_noAug": {
        "Total Training Time"  : 663.27,
        "Total Validation Time": 78.77
        },
    "SimpleAdaptiveCNN_15Epochs_noAug": {
        "Total Training Time"  : 626.09,
        "Total Validation Time": 77.79
        },
    "SimpleCNN_15Epochs_Aug": {
        "Total Training Time"  : 615.57,
        "Total Validation Time": 71.32
        },
    "SimpleAdaptiveCNN_15Epochs_Aug": {
        "Total Training Time"  : 747.35,
        "Total Validation Time": 83.39
        },

    # /cnn_trained_models/Twenty_epochs and /cnn_training_curves/Twenty_epochs
    "SimpleCNN_20Epochs_noAug": {
        "Total Training Time"  : 711.45,
        "Total Validation Time": 85.77
        },
    "SimpleAdaptiveCNN_20Epochs_noAug": {
        "Total Training Time"  : 638.99,
        "Total Validation Time": 75.98
        },

    "SimpleCNN_20Epochs_Aug": {
        "Total Training Time"  : 714.55,
        "Total Validation Time": 82.98
        },
    "SimpleAdaptiveCNN_20Epochs_Aug": {
        "Total Training Time"  : 856.05,
        "Total Validation Time": 99.3
        },

    # /cnn_trained_models/Twenty_epochs and /cnn_training_curves/Twentyfive_epochs
    "SimpleCNN_25Epochs_noAug":{
        "Total Training Time"  : 807.0,
        "Total Validation Time": 96.52
    },
    "SimpleAdaptiveCNN_25Epochs_noAug": {
        "Total Training Time"  : 997.14,
        "Total Validation Time": 128.53
    },

    "SimpleCNN_25Epochs_Aug": {
        "Total Training Time"  : 942.12,
        "Total Validation Time": 102.72
        },

    "SimpleAdaptiveCNN_25Epochs_Aug": {
        "Total Training Time"  : 845.66,
        "Total Validation Time": 93.81
        }
}


# Epochs
epochs = [10, 15, 20, 25]

# Extracted data in order of epochs above
train_times = {
    "SimpleCNN_noAug": [338.53, 663.27, 711.45, 807.0],
    "SimpleAdaptiveCNN_noAug": [338.12, 626.09, 638.99, 997.14],
    "SimpleCNN_Aug": [417.16, 615.57, 714.55, 942.12],
    "SimpleAdaptiveCNN_Aug": [454.24, 747.35, 856.05, 845.66],
}

val_times = {
    "SimpleCNN_noAug": [43.4, 78.77, 85.77, 96.52],
    "SimpleAdaptiveCNN_noAug": [42.64, 77.79, 75.98, 128.53],
    "SimpleCNN_Aug": [50.60, 71.32, 82.98, 102.72],
    "SimpleAdaptiveCNN_Aug": [51.46, 83.39, 99.3, 93.81],
}

# Training time plot
plt.figure(figsize=(7, 5))
for label, values in train_times.items():
    plt.plot(epochs, values, marker=".", label=label, linestyle=":")

plt.xlabel("Epochs")
plt.ylabel("Total Training Time (s)")
plt.title("Training Time vs Epochs")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Validation time plot
plt.figure(figsize=(7, 5))
for label, values in val_times.items():
    plt.plot(epochs, values, marker=".", label=label, linestyle=":")

plt.xlabel("Epochs")
plt.ylabel("Total Validation Time (s)")
plt.title("Validation Time vs Epochs")
plt.legend()
plt.grid(True,linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### Show here the final validation error of the models from before

**Meaning of the suffices**

/cnn_trained_models/Ten_epochs and /cnn_training_curves/Ten_epochs
+ _wo_normalization: There we did not normalize the img in Load_Cifar with std and dev
+ v1: without data augmentation
+ v2: with data augmentation

/cnn_trained_models/Fifteen_epochs and /cnn_training_curves/Fifteen_epochs
+ no suffix: without data augmentation
+ v1: with data augmentation

/cnn_trained_models/Twenty_epochs and /cnn_training_curves/Twenty_epochs
+ no suffix: without data augmentation
+ v1: with data augmentation

/cnn_trained_models/Twenty_epochs and /cnn_training_curves/Twentyfive_epochs
+ no suffix: without data augmentation
+ v1: with data augmentation

In [ ]:
in_channels = 3
num_classes = 10
input_size = 32
base_channels = 32
channel_multiplier = 2

models = {
    'SimpleCNN_10Epochs_noNorm': 'Ten_Epochs/simplecnn_cifar10_wo_normalization.pth', 'SimpleAdaptiveCNN_10Epochs_noNorm': 'Ten_Epochs/simpleadaptivecnn_cifar10_wo_normalization.pth', 
    'SimpleCNN_10Epochs_noAug': 'Ten_Epochs/simplecnn_cifar10_v1.pth', 'SimpleAdaptiveCNN_10Epochs_noAug': 'Ten_Epochs/simpleadaptivecnn_cifar10_v1.pth', 
    'SimpleCNN_10Epochs_Aug': 'Ten_Epochs/simplecnn_cifar10_v2.pth', 'SimpleAdaptiveCNN_10Epochs_Aug': 'Ten_Epochs/simpleadaptivecnn_cifar10_v2.pth', 

    'SimpleCNN_15Epochs_noAug': 'Fifteen_Epochs/simplecnn_cifar10.pth', 'SimpleAdaptiveCNN_15Epochs_noAug': 'Fifteen_Epochs/simpleadaptivecnn_cifar10.pth', 
    'SimpleCNN_15Epochs_Aug': 'Fifteen_Epochs/simplecnn_cifar10_v1.pth', 'SimpleAdaptiveCNN_15Epochs_Aug': 'Fifteen_Epochs/simpleadaptivecnn_cifar10_v1.pth', 

    'SimpleCNN_20Epochs_noAug': 'Twenty_Epochs/simplecnn_cifar10.pth', 'SimpleAdaptiveCNN_20Epochs_noAug': 'Twenty_Epochs/simpleadaptivecnn_cifar10.pth', 
    'SimpleCNN_20Epochs_Aug': 'Twenty_Epochs/simplecnn_cifar10_v1.pth', 'SimpleAdaptiveCNN_20Epochs_Aug': 'Twenty_Epochs/simpleadaptivecnn_cifar10_v1.pth', 

    'SimpleCNN_25Epochs_noAug': 'Twentyfive_Epochs/simplecnn_cifar10.pth', 'SimpleAdaptiveCNN_25Epochs_noAug': 'Twentyfive_Epochs/simpleadaptivecnn_cifar10.pth', 
    'SimpleCNN_25Epochs_Aug': 'Twentyfive_Epochs/simplecnn_cifar10_v1.pth', 'SimpleAdaptiveCNN_25Epochs_Aug': 'Twentyfive_Epochs/simpleadaptivecnn_cifar10_v1.pth'
    }


validation_error = {}
for key, value in models.items():
    if "SimpleCNN" in key:
        trained_model = SimpleCNN(in_channels=in_channels, 
                          num_classes=num_classes, 
                          input_size=input_size,
                          base_channels=base_channels, 
                          channel_multiplier=channel_multiplier,
                          dropout_rate=0.5)
    elif "SimpleAdaptiveCNN" in key:
        trained_model = SimpleAdaptiveCNN(in_channels=in_channels, 
                                  num_classes=num_classes, 
                                  input_size=input_size,
                                  base_channels=base_channels, 
                                  channel_multiplier=channel_multiplier,
                                  dropout_rate=0.5)
        
    filename = value
    trained_model.load_state_dict(torch.load(f"cnn_trained_models/CIFAR/{filename}"))

    trained_model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = trained_model(images)
            _, preds = outputs.max(1)

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    print(f"Final validation accuracy for {key}: {correct / total:.4f}")
    validation_error[key] = correct / total

In [ ]:
#print(validation_error)

epochs = [10, 15, 20, 25]
data = {
    "SimpleCNN_noAug": [0.771, 0.7693, 0.7741, 0.7732],
    "SimpleCNN_Aug": [0.7587, 0.7845, 0.7992, 0.8104],
    "SimpleAdaptiveCNN_noAug": [0.7199, 0.7493, 0.7632, 0.7649],
    "SimpleAdaptiveCNN_Aug": [0.6832, 0.7214, 0.7477, 0.7617],
}

plt.figure(figsize=(8, 5))

for label, values in data.items():
    plt.plot(epochs, values, marker=".", linestyle="--", linewidth=1, label=label)

    for x, y in zip(epochs, values): 
        plt.text(
            x, 
            y + 0.002, 
            f"{y:.3f}",
            ha="center", va="bottom", fontsize=8)
        
plt.xlabel("Epochs")
plt.ylabel("Validation Error")
plt.title("Validation Error vs Epochs")
plt.legend()
plt.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


### Plotting here the impact of dropout

Dropout tested: [0.5, 0.4, 0.3, 0.2, 0.1, 0.0]. Other "variables" set: Augmentation on and 20 Epochs

In [ ]:
dropout_model_accuracy = {
    "SimpleCNN_dropout_05": 0.7913, 
    "SimpleCNN_dropout_04": 0.7977, 
    "SimpleCNN_dropout_03": 0.7938,       
    "SimpleCNN_dropout_02": 0.8003,
    "SimpleCNN_dropout_01": 0.8146,   
    "SimpleCNN_dropout_00": 0.8002
    }

# (train, val)        
dropout_model_training_validation_time = {               
    "SimpleCNN_dropout_05": (757.07, 86.77), 
    "SimpleCNN_dropout_04": (743.74, 82.23), 
    "SimpleCNN_dropout_03": (716.00, 78.37),       
    "SimpleCNN_dropout_02": (735.72, 80.42),  
    "SimpleCNN_dropout_01": (879.49, 100.24),
    "SimpleCNN_dropout_00": (893.26, 100.07)
    }


# Extract dropout values from the name
dropout_values = np.array([int(k[-2:]) / 10 for k in dropout_model_accuracy.keys()])
accuracies = np.array(list(dropout_model_accuracy.values()))

# Accuracy Plot
plt.figure(figsize=(7, 3))
plt.plot(dropout_values, accuracies, marker=".", linestyle="--", linewidth=1)

plt.xlabel("Dropout Rate")
plt.ylabel("Accuracy")
plt.title("Accuracy of SimpleCNN (20 Epochs) vs Dropout on CIFAR-10")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(dropout_values)
plt.tight_layout()
plt.show()

# Training & Validation Time Plot
train_times = [v[0] for v in dropout_model_training_validation_time.values()]
val_times = [v[1] for v in dropout_model_training_validation_time.values()]

plt.figure(figsize=(7, 3))
plt.plot(dropout_values, train_times, marker=".", linestyle="--", linewidth=1, label="Training Time")

plt.xlabel("Dropout Rate")
plt.ylabel("Time (seconds)")
plt.title("Training vs Dropout")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(dropout_values)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
plt.plot(dropout_values, val_times, marker=".", linestyle="--", linewidth=1, label="Validation Time")

plt.xlabel("Dropout Rate")
plt.ylabel("Time (seconds)")
plt.title("Validation vs Dropout")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(dropout_values)
plt.tight_layout()
plt.show()

### Plotting here the "final" 40 Epoch run

+ in_channels = 3
+ num_classes = 10
+ input_size = 32
+ base_channels = 32
+ channel_multiplier = 2
+ epochs= 40
+ dropout_rate = 0.1

CIFAR-10 40 epochs
Total Training Time  : 1049.68s
Total Validation Time: 28.18s

Training completed.
Saved trained model to: cnn_trained_models/simplecnn_40epochs_01_dropit_cifar10.pth
Evaluating trained model on validation set...
Final validation accuracy: 0.8348

In [ ]:
epochs = [i for i in range(1,41)]

train_acc_cifar = [
    0.3977, 0.5490, 0.6212, 0.6665, 0.6888, 0.7080, 0.7243, 0.7380, 0.7448, 0.7533,
    0.7629, 0.7675, 0.7755, 0.7827, 0.7834, 0.7877, 0.7945, 0.7967, 0.8013, 0.8054,
    0.8068, 0.8087, 0.8119, 0.8172, 0.8182, 0.8182, 0.8239, 0.8240, 0.8250, 0.8292,
    0.8294, 0.8318, 0.8338, 0.8352, 0.8369, 0.8385, 0.8384, 0.8404, 0.8428, 0.8441
]

val_acc_cifar = [
    0.5488, 0.6404, 0.6831, 0.7030, 0.7324, 0.7463, 0.7557, 0.7628, 0.7720, 0.7833,
    0.7661, 0.7828, 0.7871, 0.7920, 0.7851, 0.7982, 0.8069, 0.8087, 0.8075, 0.8119,
    0.8126, 0.8091, 0.8157, 0.8199, 0.8084, 0.8162, 0.8166, 0.8076, 0.8060, 0.8158,
    0.8252, 0.8244, 0.8146, 0.8276, 0.8260, 0.8307, 0.8251, 0.8275, 0.8300, 0.8348
]

plt.figure(figsize=(7, 4))
plt.plot(epochs, train_acc_cifar, marker=".", linestyle="--", linewidth=1, label="Training Accuracy")
plt.plot(epochs, val_acc_cifar, marker=".", linestyle="--", linewidth=1, label="Validation Accuracy")

#print(val_acc_cifar.index(max(val_acc_cifar)))

plt.text(
    x=val_acc_cifar.index(max(val_acc_cifar)), 
    y=max(val_acc_cifar) - 0.1,
    s=f"Max-Validation\nAcc: {max(val_acc_cifar):.3f} \n at {val_acc_cifar.index(max(val_acc_cifar)) + 1} Epochs",
    ha="center", va="bottom", fontsize=8)
    
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracies of SimpleCNN on CIFAR-10")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

## GTSRB related Area

### Training Area

In [ ]:
# Device selection (prep for potential GPU usage in e.g Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Data Augmentation
train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomCrop(64, padding=4), # padding is ofc applied dbefore cropping randomly
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3)
])
#train_transforms = None 
val_transforms = None

training_path = "../data/GTSRB/Final_Training/Images"
testing_path = "../data/GTSRB/Final_Test/Images"

# Create datasets
train_dataset = GTSRB(base_path=training_path, split="train", transform=train_transforms)
test_dataset  = GTSRB(base_path=testing_path, split="test", transform=None)

# Class names in GTSRB
class_names = list(GTSRB_CLASSES.values())

# Create dataloaders
trainloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valloader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

# check the "batching"
images, labels = next(iter(trainloader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

#Using device: cpu
#Loading GTSRB (train) from ../data/GTSRB/Final_Training/Images...
#Loading GTSRB (test) from ../data/GTSRB/Final_Test/Images...
#Train samples: 39209
#Test samples: 12630
#Image batch shape: torch.Size([128, 3, 64, 64])
#Label batch shape: torch.Size([128])
#Image dtype: torch.float32
#Label dtype: torch.int64

In [ ]:
in_channels = 3
num_classes = len(GTSRB_CLASSES)
input_size = 64
base_channels = 32
channel_multiplier = 2
epochs=40

model_selection = { "SimpleCNN_40epochs_05_dropout": SimpleCNN(
                                in_channels=in_channels,
                                num_classes=num_classes,
                                input_size=input_size,
                                base_channels=base_channels,
                                channel_multiplier=channel_multiplier,
                                dropout_rate=0.5)}
iteration = 2
for model_name, model in model_selection.items():
    print(f"Training model: {model_name}")
    # Set the model to the device (GPU/CPU)
    model = model.to(device)
    print(model)

    # Prepare Loss function and optimizer
    criterion = nn.CrossEntropyLoss() # loss function cross entropy that will be used for calculating the loss during training so that the optimizer can update the weights accordingly (remember backpropagation!)

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    print("Starting training...")
    # Train the model
    trained_model, train_accuracies, val_accuracies = train_model(
        model=model,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epochs=epochs#*iteration
        )
    print("Training completed.")    

    # Save the trained model and check for exisiting files/dirs 
    Path("cnn_trained_models").mkdir(exist_ok=True)
    file_name = f"cnn_trained_models/{model_name.lower()}_gtsrb.pth"
    
    # Check if file exists and append version number
    base_path = Path(file_name)
    if base_path.exists():
        version = 1
        while (base_path.parent / f"{base_path.stem}_v{version}{base_path.suffix}").exists():
            version += 1
        file_name = str(base_path.parent / f"{base_path.stem}_v{version}{base_path.suffix}")

    torch.save(trained_model.state_dict(), file_name)
    print("Saved trained model to:", file_name)
    
    print("Evaluating trained model on validation set...")
    trained_model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = trained_model(images)
            _, preds = outputs.max(1)

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    print(f"Final validation accuracy: {correct / total:.4f}")

    evaluate_model(trained_model, valloader, device=device, class_names=class_names)
    iteration += 1
    del trained_model

### Analysis Area

#### Epoch Analysis

In [ ]:
epochs = [10, 15, 20, 25] 
data = {
    # [0.9612, 0.9713, 0.9745, 0.9694, 0.9747],
    "SimpleCNN_noAug": [0.9713, 0.9745, 0.9694, 0.9747],
    "SimpleCNN_Aug": [0.9477, 0.9599, 0.9537, 0.9625]
}

plt.figure(figsize=(8, 5))

for label, values in data.items():
    plt.plot(epochs, values, marker=".", linestyle="--", linewidth=1, label=label)

    for x, y in zip(epochs, values): 
        plt.text(
            x, 
            y + 0.0002, 
            f"{y:.3f}",
            ha="center", va="bottom", fontsize=8)
        
plt.xlabel("Epochs")
plt.ylabel("Validation Error")
plt.title("Validation Error vs Epochs")
plt.legend()
plt.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# Epochs
epochs = [10, 15, 20, 25] # potentially remove 5 just to be in line with the other analysis

# Extracted data in order of epochs above
train_times = {
                    # [471.32, 874.48, 1552.71, 2019.76, 2902.53],
    "SimpleCNN_noAug": [ 874.48, 1552.71, 2019.76, 2902.53],
    "SimpleCNN_Aug": [899.77, 1547.42, 2102.18, 2324.52]
}

val_times = {
                    # [73.79, 141.15, 251.09, 314.96, 456.18],
    "SimpleCNN_noAug": [141.15, 251.09, 314.96, 456.18],
    "SimpleCNN_Aug": [137.78, 248.24, 317.56, 350.46]
}

# Training time plot
plt.figure(figsize=(7, 5))
for label, values in train_times.items():
    plt.plot(epochs, values, marker=".", label=label, linestyle=":")

plt.xlabel("Epochs")
plt.ylabel("Total Training Time (s)")
plt.title("Training Time vs Epochs")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Validation time plot
plt.figure(figsize=(7, 5))
for label, values in val_times.items():
    plt.plot(epochs, values, marker=".", label=label, linestyle=":")

plt.xlabel("Epochs")
plt.ylabel("Total Validation Time (s)")
plt.title("Validation Time vs Epochs")
plt.legend()
plt.grid(True,linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

#### Dropout Analysis

In [ ]:
dropout_model_accuracy_gtsrb = {
    "SimpleCNN_dropout_05": 0.9564, 
    "SimpleCNN_dropout_04": 0.9530, 
    "SimpleCNN_dropout_03": 0.9444,       
    "SimpleCNN_dropout_02": 0.9457,
    "SimpleCNN_dropout_01": 0.9439,   
    "SimpleCNN_dropout_00": 0.9338
    }

dropout_model_accuracy_cifar = {
    "SimpleCNN_dropout_05": 0.7913, 
    "SimpleCNN_dropout_04": 0.7977, 
    "SimpleCNN_dropout_03": 0.7938,       
    "SimpleCNN_dropout_02": 0.8003,
    "SimpleCNN_dropout_01": 0.8146,   
    "SimpleCNN_dropout_00": 0.8002
    }


# (train, val)        
dropout_model_training_validation_time = {               
    "SimpleCNN_dropout_05": (2003.03, 318.62), 
    "SimpleCNN_dropout_04": (2271.84, 355.33), 
    "SimpleCNN_dropout_03": (2137.51, 321.35),       
    "SimpleCNN_dropout_02": (1994.74, 305.32),  
    "SimpleCNN_dropout_01": (2246.31, 345.36),
    "SimpleCNN_dropout_00": (2115.72, 330.05)
    }


# Extract dropout values from the name
dropout_values_gtsrb = np.array([int(k[-2:]) / 10 for k in dropout_model_accuracy_gtsrb.keys()])
accuracies_gtsrb = np.array(list(dropout_model_accuracy_gtsrb.values()))

dropout_values_cifar = np.array([int(k[-2:]) / 10 for k in dropout_model_accuracy_cifar.keys()])
accuracies_cifar = np.array(list(dropout_model_accuracy_cifar.values()))

# Accuracy Plot
plt.figure(figsize=(7, 3))
plt.plot(dropout_values_gtsrb, accuracies_gtsrb,  marker=".", linestyle="--", linewidth=1) #label= "gtsrb",
#plt.plot(dropout_values_cifar, accuracies_cifar, label= "cifar", marker=".", linestyle="--", linewidth=1)

plt.xlabel("Dropout Rate")
plt.ylabel("Accuracy")
plt.title("Accuracy of SimpleCNN (20 Epochs) vs Dropout on GTSRB")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(dropout_values)
#plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
#plt.plot(dropout_values_gtsrb, accuracies_gtsrb, label= "gtsrb", marker=".", linestyle="--", linewidth=1)
plt.plot(dropout_values_cifar, accuracies_cifar,  marker=".", linestyle="--", linewidth=1) #label= "cifar",

plt.xlabel("Dropout Rate")
plt.ylabel("Accuracy")
plt.title("Accuracy of SimpleCNN (20 Epochs) vs Dropout on CIFAR-10")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(dropout_values)
#plt.legend()
plt.tight_layout()
plt.show()

# Training & Validation Time Plot
train_times = [v[0] for v in dropout_model_training_validation_time.values()]
val_times = [v[1] for v in dropout_model_training_validation_time.values()]

plt.figure(figsize=(7, 3))
plt.plot(dropout_values, train_times, marker=".", linestyle="--", linewidth=1, label="Training Time")

plt.xlabel("Dropout Rate")
plt.ylabel("Time (seconds)")
plt.title("Training vs Dropout")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(dropout_values)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
plt.plot(dropout_values, val_times, marker=".", linestyle="--", linewidth=1, label="Validation Time")

plt.xlabel("Dropout Rate")
plt.ylabel("Time (seconds)")
plt.title("Validation vs Dropout")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(dropout_values)
plt.tight_layout()
plt.show()

### Plotting here the "final" 40 Epoch run

+ in_channels = 3
+ num_classes = len(GTSRB_CLASSES)
+ input_size = 64
+ base_channels = 32
+ channel_multiplier = 2
+ epochs=40
+ dropout_rate = 0.5

GTSRB 40 epochs
Total Training Time  : 1301.33s
Total Validation Time: 129.73s

Training completed.
Saved trained model to: cnn_trained_models/simplecnn_40epochs_05_dropout_gtsrb.pth
Evaluating trained model on validation set...
Final validation accuracy: 0.9565

In [ ]:
epochs = [i for i in range(1,41)]

train_acc_gtsrb = [
    0.4096, 0.7244, 0.8320, 0.8836, 0.9101, 0.9253, 0.9385, 0.9461, 0.9504, 0.9572,
    0.9597, 0.9622, 0.9675, 0.9663, 0.9688, 0.9713, 0.9717, 0.9745, 0.9746, 0.9773,
    0.9767, 0.9767, 0.9775, 0.9798, 0.9788, 0.9816, 0.9806, 0.9827, 0.9831, 0.9823,
    0.9833, 0.9835, 0.9832, 0.9829, 0.9832, 0.9859, 0.9854, 0.9859, 0.9851, 0.9861
]
val_acc_gtsrb = [
    0.7071, 0.8408, 0.9029, 0.9220, 0.9264, 0.9294, 0.9439, 0.9437, 0.9546, 0.9487,
    0.9416, 0.9500, 0.9508, 0.9572, 0.9549, 0.9510, 0.9580, 0.9534, 0.9539, 0.9599,
    0.9566, 0.9561, 0.9561, 0.9552, 0.9573, 0.9607, 0.9637, 0.9564, 0.9574, 0.9569,
    0.9613, 0.9586, 0.9531, 0.9594, 0.9634, 0.9603, 0.9550, 0.9603, 0.9496, 0.9565
]

plt.figure(figsize=(7, 4))
plt.plot(epochs, train_acc_gtsrb, marker=".", linestyle="--", linewidth=1, label="Training Accuracy")
plt.plot(epochs, val_acc_gtsrb, marker=".", linestyle="--", linewidth=1, label="Validation Accuracy")

#print(val_acc_cifar.index(max(val_acc_cifar)))

plt.text(
    x=val_acc_gtsrb.index(max(val_acc_gtsrb)) + 1, 
    y=max(val_acc_gtsrb) - 0.1,
    s=f"Max-Validation\nAcc: {max(val_acc_gtsrb):.3f} \n at {val_acc_gtsrb.index(max(val_acc_gtsrb)) + 1} Epochs",
    ha="center", va="bottom", fontsize=8)
    
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracies of SimpleCNN on GTSRB")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

## Playground Area

In [ ]:
# Investigate whether augmentation at least reduced the loss for classes that are dependend on the direction (go right signs and such)
# Theory is now that augmentation didnt really improve anything here for GTSRB as it might be invariant really (circular signs e.g.)

in_channels = 3
num_classes = len(GTSRB_CLASSES)
input_size = 64
base_channels = 32
channel_multiplier = 2

trained_model_aug = SimpleCNN(in_channels=in_channels, 
                    num_classes=num_classes, 
                    input_size=input_size,
                    base_channels=base_channels, 
                    channel_multiplier=channel_multiplier,
                    dropout_rate=0.5)

filename = "cnn_trained_models/GTSRB/SimpleCNN_Epochs_withAug/simplecnn_epoch_25_gtsrb.pth"
trained_model_aug.load_state_dict(torch.load(filename))


trained_model_noaug = SimpleCNN(in_channels=in_channels, 
                    num_classes=num_classes, 
                    input_size=input_size,
                    base_channels=base_channels, 
                    channel_multiplier=channel_multiplier,
                    dropout_rate=0.5)

filename = "cnn_trained_models/GTSRB/SimpleCNN_Epochs_noAug/simplecnn_epoch_25_gtsrb.pth"
trained_model_noaug.load_state_dict(torch.load(filename))

evaluate_model(trained_model_aug, valloader, device=device, class_names=class_names)
evaluate_model(trained_model_noaug, valloader, device=device, class_names=class_names)
